In [1]:
# best-user-chosen mode + user-chosen thresholds + labels + file open dialog
import cv2
import torch
import torch.nn as nn
import numpy as np
from ultralytics import YOLO
import os
import time
from collections import deque
from tkinter import Tk
from tkinter.filedialog import askopenfilename  # file dialog

# ====== CONFIG (defaults, can be overridden by user) ======
DENSITY_THRESHOLD = 8.0  # will be overwritten by user input
SPEED_THRESHOLD = 5.0  # will be overwritten by user input
YOLO_MODEL_PATH = "yolov8n.pt"
YOLO_CONF_THRESH = 0.10  # more sensitive
CALIBRATION_FACTOR = 1.0  # will be auto-updated
CAPACITY_MAX = 100.0  # assumed safe capacity for occupancy %
CALIB_FRAMES = 60  # frames used for auto-calibration factor only


# ====== MC-CNN MODEL ======
class MC_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.column1 = nn.Sequential(
            nn.Conv2d(3, 8, 9, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 7, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 7, padding='same'),
            nn.ReLU(),
            nn.Conv2d(32, 16, 7, padding='same'),
            nn.ReLU(),
            nn.Conv2d(16, 8, 7, padding='same'),
            nn.ReLU(),
        )
        self.column2 = nn.Sequential(
            nn.Conv2d(3, 10, 7, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(10, 20, 5, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(20, 40, 5, padding='same'),
            nn.ReLU(),
            nn.Conv2d(40, 20, 5, padding='same'),
            nn.ReLU(),
            nn.Conv2d(20, 10, 5, padding='same'),
            nn.ReLU(),
        )
        self.column3 = nn.Sequential(
            nn.Conv2d(3, 12, 5, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(12, 24, 3, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(24, 48, 3, padding='same'),
            nn.ReLU(),
            nn.Conv2d(48, 24, 3, padding='same'),
            nn.ReLU(),
            nn.Conv2d(24, 12, 3, padding='same'),
            nn.ReLU(),
        )
        self.fusion_layer = nn.Sequential(
            nn.Conv2d(30, 1, 1, padding=0),
        )

    def forward(self, x):
        x1 = self.column1(x)
        x2 = self.column2(x)
        x3 = self.column3(x)
        x = torch.cat((x1, x2, x3), dim=1)
        x = self.fusion_layer(x)
        return x


def preprocess_patch(patch_bgr, target_size=(768, 512)):
    rgb = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, target_size)
    img = resized.astype(np.float32) / 255.0
    img = np.transpose(img, (2, 0, 1))
    tensor = torch.from_numpy(img).unsqueeze(0)
    return tensor


def density_to_heatmap(dmap_np, out_w, out_h):
    if dmap_np.max() > 0:
        norm = dmap_np / (dmap_np.max() + 1e-6)
    else:
        norm = dmap_np
    heat = (norm * 255).astype(np.uint8)
    heat_color = cv2.applyColorMap(heat, cv2.COLORMAP_JET)
    heat_color = cv2.resize(heat_color, (out_w, out_h))
    return heat_color


def format_time(seconds):
    m = int(seconds // 60)
    s = int(seconds % 60)
    return f"{m:02d}:{s:02d}"


def auto_calibrate(mccnn, yolo, cap, device, num_frames=60):
    yolo_total = 0.0
    mcnn_raw_total = 0.0
    frames_used = 0

    pos0 = cap.get(cv2.CAP_PROP_POS_FRAMES)

    while frames_used < num_frames:
        ret, frame = cap.read()
        if not ret:
            break

        results = yolo(frame, conf=YOLO_CONF_THRESH, classes=[0], imgsz=960, verbose=False)
        boxes = results[0].boxes if results[0].boxes is not None else []
        yolo_count = len(boxes)
        yolo_total += yolo_count

        inp = preprocess_patch(frame).to(device)
        with torch.no_grad():
            dmap = mccnn(inp)
        mcnn_raw = float(dmap.sum().item())
        mcnn_raw_total += mcnn_raw

        frames_used += 1

    cap.set(cv2.CAP_PROP_POS_FRAMES, pos0)

    if frames_used == 0:
        print("Calibration: no frames read, keeping default factor=1.0")
        return CALIBRATION_FACTOR

    print(f"Calibration: frames_used={frames_used}, YOLO_total={yolo_total:.1f}, MCNN_raw_total={mcnn_raw_total:.3f}")

    if mcnn_raw_total > 0 and yolo_total > 0:
        raw_factor = yolo_total / (mcnn_raw_total + 1e-6)
        calib_factor = raw_factor * 15.0
        print(f"Calibration: raw_factor={raw_factor:.6f}, calib_factor={calib_factor:.6f}")
        return calib_factor
    else:
        print("Calibration: zero totals, using default factor=1.0")
        return CALIBRATION_FACTOR


def ask_user_mode():
    print("Select crowd type for this video:")
    print("  1 - Sparse crowd (use YOLO for density)")
    print("  2 - Dense crowd  (use MCNN for density)")
    print("  3 - Unknown      (display both YOLO and MCNN)")
    mode = None
    while mode not in {"1", "2", "3"}:
        mode = input("Enter 1 / 2 / 3: ").strip()
    if mode == "1":
        return "SPARSE"
    elif mode == "2":
        return "DENSE"
    else:
        return "UNKNOWN"


def ask_user_thresholds():
    global DENSITY_THRESHOLD, SPEED_THRESHOLD

    try:
        dt_str = input(f"Enter density threshold (default {DENSITY_THRESHOLD}): ").strip()
        if dt_str != "":
            DENSITY_THRESHOLD = float(dt_str)
    except ValueError:
        print("Invalid density threshold, keeping default:", DENSITY_THRESHOLD)

    try:
        st_str = input(f"Enter speed threshold (default {SPEED_THRESHOLD}): ").strip()
        if st_str != "":
            SPEED_THRESHOLD = float(st_str)
    except ValueError:
        print("Invalid speed threshold, keeping default:", SPEED_THRESHOLD)

    print(f"Using DENSITY_THRESHOLD={DENSITY_THRESHOLD}, SPEED_THRESHOLD={SPEED_THRESHOLD}")


def main():
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    print("Using device:", device)

    # ----- video file selection dialog -----
    root = Tk()
    root.withdraw()  # hide main Tk window
    print("Please select a video file...")
    video_path = askopenfilename(
        title="Select input video",
        filetypes=[("Video files", "*.mp4 *.avi *.mkv *.mov"), ("All files", "*.*")]
    )
    root.destroy()

    if not video_path:
        print("No video selected. Exiting.")
        return

    print("Selected video:", video_path)

    mccnn = MC_CNN().to(device)
    mccnn.load_state_dict(torch.load("crowd_counting.pth", map_location=device), strict=False)
    mccnn.eval()
    print("MC-CNN model loaded.")

    yolo = YOLO(YOLO_MODEL_PATH)
    print("YOLOv8 model loaded.")

    output_video_path = "output_density_yolo_user_mode_thresh_labels.mp4"

    if not os.path.exists(video_path):
        print(f" File not found: {video_path}")
        return

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("Error opening video file.")
        return

    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 20.0

    user_mode = ask_user_mode()
    ask_user_thresholds()
    print("User selected mode:", user_mode)

    global CALIBRATION_FACTOR
    CALIBRATION_FACTOR = auto_calibrate(mccnn, yolo, cap, device, num_frames=CALIB_FRAMES)

    codecs = [cv2.VideoWriter_fourcc(*'mp4v'), cv2.VideoWriter_fourcc(*'XVID')]
    out = None
    for codec in codecs:
        out = cv2.VideoWriter(output_video_path, codec, fps, (w, h))
        if out.isOpened():
            break
        out.release()
    if not out or not out.isOpened():
        output_video_path = output_video_path.replace('.mp4', '.avi')
        out = cv2.VideoWriter(output_video_path, cv2.VideoWriter_fourcc(*'XVID'), fps, (w, h))

    prev_centers = {}
    frame_idx = 0
    start_time = time.time()
    primary_hist = deque(maxlen=30)

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        overlay = frame.copy()

        # MC-CNN
        orig_h, orig_w = frame.shape[:2]
        inp = preprocess_patch(frame).to(device)
        with torch.no_grad():
            dmap = mccnn(inp)
        dnp = dmap.squeeze().cpu().numpy()
        mcnn_raw = float(dnp.sum())
        mcnn_count = max(0.0, mcnn_raw * CALIBRATION_FACTOR)
        heatmap = density_to_heatmap(dnp, orig_w, orig_h)
        overlay = cv2.addWeighted(overlay, 0.6, heatmap, 0.4, 0)

        # YOLO
        results = yolo(frame, conf=YOLO_CONF_THRESH, classes=[0], imgsz=960, verbose=False)
        boxes = results[0].boxes if results[0].boxes is not None else []
        speeds = []
        new_prev_centers = {}
        total_persons = len(boxes)

        for idx, box in enumerate(boxes):
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
            cx = int((x1 + x2) / 2)
            cy = int((y1 + y2) / 2)

            track_id = idx
            if track_id in prev_centers:
                px, py = prev_centers[track_id]
                speed = float(np.sqrt((cx - px) ** 2 + (cy - py) ** 2))
            else:
                speed = 0.0

            speeds.append(speed)
            new_prev_centers[track_id] = (cx, cy)
            cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(overlay, f"v={speed:.1f}", (x1, y1 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)

        prev_centers = new_prev_centers
        avg_speed = float(np.mean(speeds)) if speeds else 0.0

        # Primary count selection
        if user_mode == "SPARSE":
            primary_count = float(total_persons)
            primary_source = "YOLO"
        elif user_mode == "DENSE":
            primary_count = mcnn_count
            primary_source = "MCNN"
        else:
            primary_count = float(total_persons)
            primary_source = "YOLO+MCNN"

        primary_hist.append(primary_count)
        primary_smooth = sum(primary_hist) / len(primary_hist)

        occupancy = 0.0
        if CAPACITY_MAX > 0:
            occupancy = min(100.0, (primary_count / CAPACITY_MAX) * 100.0)

        # Risk / stampede
        if avg_speed < 1.0:
            risk_text = "LOW RISK" if primary_count <= DENSITY_THRESHOLD else "MEDIUM RISK"
            risk_color = (0, 255, 0) if primary_count <= DENSITY_THRESHOLD else (0, 255, 255)
            stampede_text = "NOT STAMPEDE PRONE"
            stampede_color = (0, 255, 0)
        else:
            if primary_count > DENSITY_THRESHOLD and avg_speed >= SPEED_THRESHOLD:
                risk_text = "HIGH RISK"
                risk_color = (0, 0, 255)
                stampede_text = "STAMPede PRONE"
                stampede_color = (0, 0, 255)
            elif primary_count > DENSITY_THRESHOLD:
                risk_text = "MEDIUM RISK"
                risk_color = (0, 255, 255)
                stampede_text = "NOT STAMPEDE PRONE"
                stampede_color = (0, 255, 0)
            else:
                risk_text = "LOW RISK"
                risk_color = (0, 255, 0)
                stampede_text = "NOT STAMPEDE PRONE"
                stampede_color = (0, 255, 0)

        print(f"frame={frame_idx}, user_mode={user_mode}, "
              f"mcnn_raw={mcnn_raw:.3f}, mcnn_count={mcnn_count:.3f}, "
              f"yolo_persons={total_persons}, primary={primary_count:.3f}({primary_source}), "
              f"avg_speed={avg_speed:.2f}, occ={occupancy:.1f}%, risk={risk_text}")

        # Overlays
        red = (0, 0, 255)
        green = (0, 255, 0)
        yellow = (0, 255, 255)
        cyan = (255, 255, 0)
        white = (255, 255, 255)

        cv2.putText(overlay, f"Mode: {user_mode}", (20, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, white, 2)

        if user_mode == "SPARSE":
            cv2.putText(overlay, f"Crowd density (YOLO): {total_persons}", (20, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, red, 2)
        elif user_mode == "DENSE":
            cv2.putText(overlay, f"Crowd density (MCNN): {mcnn_count:.1f}", (20, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, red, 2)
        else:
            cv2.putText(overlay, f"Crowd density (MCNN): {mcnn_count:.1f}", (20, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, red, 2)
            cv2.putText(overlay, f"Crowd density (YOLO): {total_persons}", (20, 90),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, red, 2)

        cv2.putText(overlay, f"Avg speed: {avg_speed:.1f}", (20, 120),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, red, 2)

        cv2.putText(overlay, f"Primary density: {primary_count:.1f} ({primary_source})", (20, 150),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, cyan, 2)

        if primary_count > DENSITY_THRESHOLD:
            cv2.putText(overlay, "OVER-CROWDING WARNING", (20, 190),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, red, 2)
        else:
            cv2.putText(overlay, "NOT OVERCROWDED", (20, 190),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, green, 2)

        cv2.putText(overlay, stampede_text, (20, 225),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, stampede_color, 2)

        cv2.putText(overlay, f"Occupancy: {occupancy:.0f}%", (20, 260),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, yellow, 2)

        cv2.putText(overlay, f"Primary (avg): {primary_smooth:.1f}", (20, 295),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, cyan, 2)

        cv2.putText(overlay, f"Risk: {risk_text}", (20, 330),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, risk_color, 2)

        elapsed_sec = time.time() - start_time
        time_str = format_time(elapsed_sec)
        cv2.putText(overlay, f"Time: {time_str}", (w - 220, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, white, 2)

        frame_idx += 1

        out.write(overlay)
        cv2.imshow("Density + YOLO (User Mode + Thresholds + Labels)", overlay)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print("Saved:", output_video_path)


if __name__ == "__main__":
    main()


Using device: cpu
Please select a video file...
Selected video: C:/Users/lekha/OneDrive/Videos/test video/istockphoto-2149553121-640_adpp_is.mp4
MC-CNN model loaded.
YOLOv8 model loaded.
Select crowd type for this video:
  1 - Sparse crowd (use YOLO for density)
  2 - Dense crowd  (use MCNN for density)
  3 - Unknown      (display both YOLO and MCNN)
Using DENSITY_THRESHOLD=123.0, SPEED_THRESHOLD=23.0
User selected mode: DENSE
Calibration: frames_used=60, YOLO_total=1178.0, MCNN_raw_total=2757.224
Calibration: raw_factor=0.427241, calib_factor=6.408620
frame=0, user_mode=DENSE, mcnn_raw=45.956, mcnn_count=294.514, yolo_persons=19, primary=294.514(MCNN), avg_speed=0.00, occ=100.0%, risk=MEDIUM RISK
frame=1, user_mode=DENSE, mcnn_raw=45.954, mcnn_count=294.501, yolo_persons=23, primary=294.501(MCNN), avg_speed=136.13, occ=100.0%, risk=HIGH RISK
frame=2, user_mode=DENSE, mcnn_raw=45.966, mcnn_count=294.576, yolo_persons=27, primary=294.576(MCNN), avg_speed=192.21, occ=100.0%, risk=HIGH RI